# Dental AI — Kaggle 3D görsel testi

Bu notebook `work/vision48-3d-ready` dalındaki panoramik tabanlı 3D görüntüleyiciyi geçici olarak açar. **Kaggle Internet açık olmalı; hız için GPU T4 önerilir.** Oluşan adres herkese açık ve geçicidir. Kimliği belirlenebilir hasta verisi yüklemeyin. Bu görünüm CBCT veya gerçek medikal hacim değildir.

In [ ]:
from pathlib import Path
import hashlib, os, re, subprocess, sys, time
import requests
from IPython.display import HTML, display

REPOSITORY = 'https://github.com/dentalaidestek/dental-ai-v02.git'
BRANCH = 'work/vision48-3d-ready'
TEST_COMMIT = '16f24f63eb655fd2312b56c266b5c724c4b62313'
WORKDIR = Path('/kaggle/working/dental-ai-v02')
print('Test commit:', TEST_COMMIT)

In [ ]:
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--detach', TEST_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'], text=True).strip()
assert head == TEST_COMMIT, (head, TEST_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn[standard]', 'python-multipart', 'ultralytics', 'huggingface_hub', 'scikit-image'], check=True)
print('Kod ve bağımlılıklar hazır.')

In [ ]:
MODEL_DIR = WORKDIR / 'models' / 'vision'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
assets = [
    ('vision-model-v1', 'YOLOv11x-seg.pt', '2d0c07b878e9f9730eb845a901e85abf3535d49cf4034d2246592f76201ac8af'),
    ('vision-findings9-v1', 'YOLO26_Dental_Findings_9.pt', '8070505857f354aae4f18bf621b9b79f0e3b48a2904e44f57a2bc598ed692849'),
    ('vision-impacted-v1', 'OralGuard_Impacted.pt', 'c3303656e72ede3f3d3229e58b2da276448fa204046d3e7a11e13f4d77bc723a'),
]
for tag, name, expected in assets:
    target = MODEL_DIR / name
    if not target.exists():
        url = f'https://github.com/dentalaidestek/dental-ai-v02/releases/download/{tag}/{name}'
        with requests.get(url, stream=True, timeout=300) as response:
            response.raise_for_status()
            with target.open('wb') as output:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk: output.write(chunk)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    assert actual == expected, f'{name}: SHA256 uyuşmuyor'
    print(f'OK {name} ({target.stat().st_size/1024/1024:.1f} MB)')

In [ ]:
checks = [
    ['node', '--check', 'vision_service/templates/viewer_v3.js'],
    ['node', '--check', 'vision_service/templates/viewer_v3_finalfix.js'],
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_3d*.py', '-v'],
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_root_filling_recovery.py', '-v'],
]
for command in checks:
    subprocess.run(command, cwd=WORKDIR, check=True)
print('3D kod kontrolleri geçti.')

In [ ]:
server_log = open('/kaggle/working/dental_3d_server.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'vision_service.anatomy3d_direct_app:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=WORKDIR, stdout=server_log, stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=2).ok: break
    except requests.RequestException: pass
    time.sleep(1)
else:
    raise RuntimeError(Path('/kaggle/working/dental_3d_server.log').read_text()[-4000:])
print('Yerel 3D sunucusu hazır.')

In [ ]:
cloudflared = Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with cloudflared.open('wb') as output:
            for chunk in response.iter_content(1024 * 1024):
                if chunk: output.write(chunk)
    cloudflared.chmod(0o755)
tunnel_log_path = Path('/kaggle/working/dental_3d_tunnel.log')
tunnel_log = tunnel_log_path.open('w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(60):
    time.sleep(1)
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tunnel_log_path.read_text(errors='ignore'))
    if match:
        public_url = match.group(0) + '/viewer'
        break
if not public_url:
    raise RuntimeError(tunnel_log_path.read_text(errors='ignore')[-4000:])
display(HTML(f'<h3><a href="{public_url}" target="_blank">3D TEST EKRANINI AÇ</a></h3><p>Link yalnız bu Kaggle oturumu çalışırken açıktır.</p>'))
print(public_url)

## Kontrol listesi

1. Test ekranını açın ve kimlik bilgisi içermeyen bir panoramik yükleyin.
2. Diş sayısı ve eksik dişleri panoramikle karşılaştırın.
3. Dişlerin uzun eksenlerinin filme benzerliğini kontrol edin.
4. Gömülü diş varsa açı ve yaklaşık konumunu kontrol edin.
5. Pembe mandibular kanalların mandibula içinde kalıp kalmadığına bakın.
6. Döndürme, yakınlaştırma ve dişe dokunarak ayrıntı ekranını deneyin.

Panoramikten bukkolingual derinlik ölçülmediği için önden/arkadan derinlik hasta anatomisi olarak değerlendirilmemelidir.

In [ ]:
# Test bitince geçici bağlantıyı kapatın.
for process_name in ('tunnel', 'server'):
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
print('Geçici test sunucusu kapatıldı.')